# RunAI — Pipeline Principal

Ejecuta toda la pipeline del TFM en orden. Cada sección es independiente y puede relanzarse por separado.

| Paso | Notebook | Descripción |
|------|----------|-------------|
| 0 | — | Setup: dependencias y parche del loader |
| 1 | 01_eda | Análisis exploratorio |
| 2 | 02_preprocessing | Limpieza + features + TRIMP → parquet |
| 3 | 04_modeling | Entrenamiento de modelos (~1-4h en CPU) |
| 4 | 05_evaluation | Métricas y comparativa |
| 5 | 06_shap | Interpretabilidad SHAP |
| 6 | 07_acwr | Análisis ACWR |


## Paso 0 — Setup
Parchea `data_loader.py` para soportar el formato de `endomondoHR.json` e instala dependencias faltantes.

In [ ]:
import ast, json, subprocess, sys, importlib, logging
from pathlib import Path

# Raíz del proyecto (un nivel arriba de notebooks/)
ROOT = Path().resolve().parent
if not (ROOT / 'src').exists():
    ROOT = Path().resolve()   # fallback si se lanza desde la raíz
sys.path.insert(0, str(ROOT))
print(f'Raíz del proyecto : {ROOT}')

# ── Parche data_loader.py ────────────────────────────────────────────────────
LOADER_PATH = ROOT / 'src' / 'data_loader.py'
src = LOADER_PATH.read_text(encoding='utf-8')

needs_patch = 'ast.literal_eval' not in src
if needs_patch:
    src = src.replace('import json', 'import ast\nimport json', 1)
    src = src.replace(
        'except json.JSONDecodeError as exc:\n                logger.warning("Línea %d malformada, se omite: %s", i, exc)',
        'except json.JSONDecodeError:\n                try:\n                    records.append(ast.literal_eval(line))\n                except Exception as exc:\n                    logger.warning("Línea %d malformada, se omite: %s", i, exc)'
    )
    LOADER_PATH.write_text(src, encoding='utf-8')
    print('✓ data_loader.py parcheado (ast.literal_eval añadido)')
else:
    print('✓ data_loader.py ya tiene el parche')

# ── Instalar pyarrow ─────────────────────────────────────────────────────────
try:
    import pyarrow
    print(f'✓ pyarrow {pyarrow.__version__} disponible')
except ImportError:
    print('Instalando pyarrow...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'pyarrow', '-q'])
    print('✓ pyarrow instalado')

# Recargar data_loader por si estaba cacheado
for mod in [k for k in sys.modules if 'data_loader' in k]:
    del sys.modules[mod]

print('\n✓ Setup completado')

## Pasos 1-6 — Ejecución de notebooks

La celda siguiente ejecuta cada notebook en orden y muestra el resultado. Si un paso falla, se detiene y muestra el error. Puedes relanzar desde cualquier paso editando `START_FROM`.

In [ ]:
# -- Configuración ------------------------------------------------------------
# Cambia START_FROM para relanzar desde un paso concreto (1-6)
START_FROM = 1

# Cambia a True para ejecutar solo un notebook concreto
RUN_ONLY = None   # ej: '02_preprocessing'

NOTEBOOKS = [
    (1, '01_eda.ipynb',                 'Análisis exploratorio'),
    (2, '02_preprocessing.ipynb',       'Limpieza + features + TRIMP → parquet'),
    (3, '04_modeling.ipynb',            'Entrenamiento de modelos  [puede tardar ~1-4h]'),
    (4, '05_evaluation.ipynb',          'Evaluación y métricas'),
    (5, '06_shap_interpretability.ipynb', 'Interpretabilidad SHAP'),
    (6, '07_acwr_analysis.ipynb',       'Análisis ACWR'),
]

print('Notebooks a ejecutar:')
for step, nb, desc in NOTEBOOKS:
    skip = step < START_FROM or (RUN_ONLY and RUN_ONLY not in nb)
    mark = '--' if skip else '>'
    print(f'  {mark} Paso {step}: {nb:40s}  {desc}')

In [ ]:
import time

NB_DIR = ROOT / 'notebooks'
results = []
sep = '─' * 65

for step, nb_file, desc in NOTEBOOKS:
    if step < START_FROM:
        print(f'  Paso {step} ({nb_file}) - saltado')
        continue
    if RUN_ONLY and RUN_ONLY not in nb_file:
        print(f'  Paso {step} ({nb_file}) - saltado')
        continue

    nb_path = NB_DIR / nb_file
    if not nb_path.exists():
        print(f'  Paso {step} ({nb_file}) - archivo no encontrado, saltando')
        continue

    print()
    print(sep)
    print(f'>  Paso {step}: {nb_file}')
    print(f'   {desc}')
    print(sep)

    t0 = time.time()
    proc = subprocess.run(
        [
            sys.executable, '-m', 'nbconvert',
            '--to', 'notebook',
            '--execute',
            '--inplace',
            '--ExecutePreprocessor.timeout=14400',
            '--ExecutePreprocessor.kernel_name=python3',
            str(nb_path),
        ],
        capture_output=True,
        text=True,
    )
    elapsed = time.time() - t0
    mins = int(elapsed // 60)
    secs = int(elapsed % 60)

    if proc.returncode == 0:
        print(f'OK  Completado en {mins}m {secs}s')
        results.append((step, nb_file, 'OK', elapsed))
    else:
        stderr_lines = (proc.stderr or '').splitlines()
        relevant = [l for l in stderr_lines if 'Error' in l or 'Traceback' in l]
        print(f'ERROR  (tiempo: {mins}m {secs}s)')
        print(chr(10).join(relevant[-20:]) if relevant else proc.stderr[-2000:])
        results.append((step, nb_file, 'ERROR', elapsed))
        print('Pipeline detenida. Ajusta START_FROM para reintentar.')
        break

print('=' * 65)
print('RESUMEN')
print('=' * 65)
for step, nb, status, elapsed in results:
    icon = 'OK' if status == 'OK' else 'ERR'
    mins = int(elapsed // 60)
    secs = int(elapsed % 60)
    print(f'  {icon}  Paso {step}  {nb:42s}  {mins}m {secs}s')

all_ok = all(s == 'OK' for _, _, s, _ in results)
if all_ok and results:
    print('Pipeline completa.')


## Relanzar un paso concreto

Si necesitas repetir solo un notebook (p. ej. tras corregir un error):

In [ ]:
# Descomenta y ejecuta para relanzar un notebook individual
# RUN_ONLY = '02_preprocessing'
# START_FROM = 1

# O lanza directamente:
# nb = ROOT / 'notebooks' / '02_preprocessing.ipynb'
# subprocess.run([sys.executable, '-m', 'nbconvert', '--to', 'notebook',
#                 '--execute', '--inplace', str(nb)])